In [2]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import csv
import pandas as pd

In [ ]:
# crawl through every item's webpage

visited_urls = set()
BASE_DOMAIN = "www.musikwissenschaft.uni-wuerzburg.de"
BASE_PATH = "https://www.musikwissenschaft.uni-wuerzburg.de/musikinstrumente/bestand/inventarliste/"
scraped_text = []

def scrape(url):
    if url in visited_urls:
        return
    visited_urls.add(url)


    r = requests.get(url, timeout=10)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")
    scraped_text.append(url)
    for div in soup.find_all("p", "bodytext"): # idea: select divs that contain p bodytext elements?
        text = div.get_text()
        print(text)
        scraped_text.append(text)

    for tag in soup.find_all("a", href=True):

        next_url = urljoin(url, tag["href"]).split("#")[0]

        if (
            urlparse(next_url).netloc == BASE_DOMAIN
            and next_url.startswith(BASE_PATH)
            and next_url not in visited_urls
        ):
            scrape(next_url)

start_url = "https://www.musikwissenschaft.uni-wuerzburg.de/musikinstrumente/bestand/inventarliste/"
scrape(start_url)

with open("uncleaned_itemsTest.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(scraped_text))


In [ ]:
with open("uncleaned_items.txt", "r", encoding="utf-8") as f:
    text = f.read()


items = {}

for part in text.split("https://www.musikwissenschaft.uni-wuerzburg.de/musikinstrumente/bestand/inventarliste/"):
    url, _, rest = part.partition( "\n")  # split the part into rest of url and the rest of the text
    item_id = url.split("/")[0]  # extract the item id from the url
    items[item_id] = (rest.strip())  # store the rest of the text in the dictionary with the item id as key

# first entry is empty, remove it
items.pop("", None)

# dict to csv
with open("uncleaned_items.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "text"])
    w.writerows(items.items())

In [ ]:
# merge with inventory_numbers.csv
inventory_df = pd.read_csv("data/inventory_numbers.csv", sep=";", encoding="utf-8", header=None, names=["id", "name", "description"])
uncleaned_df = pd.read_csv("uncleaned_items.csv", encoding="utf-8")

# create keys for merging 
uncleaned_df["key"] = uncleaned_df["id"].str.split("-").str[0] # ate1-bode-melochord -> ate1
inventory_df["key"] = inventory_df["id"].str.lower().str.replace(" ", "") # Ate 1 -> ate1

# merge 
merged_df = pd.merge(inventory_df, uncleaned_df, on="key", how="left")
merged_df = merged_df.rename(columns={"id_x": "url_id", "id_y": "inventory_id", "description":"inventory_list_description"}) # rename for clarity

# save
merged_df.to_csv("uncleaned_items_v2.csv", sep=";", encoding="utf-8", index=False, header=True)

---

In [ ]:
# Extract indexed instruments within the collection

visited_urls = set()
BASE_DOMAIN = "www.musikwissenschaft.uni-wuerzburg.de"
BASE_PATH = "https://www.musikwissenschaft.uni-wuerzburg.de/musikinstrumente/bestand/inventarliste/"
scraped_text = []

def scrape(url):
    if url in visited_urls:
        return
    visited_urls.add(url)
    print(url)

    r = requests.get(url, timeout=10)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    for p in soup.find_all("p"):
        text = p.get_text(strip=True, separator=";")
        print(text)
        scraped_text.append(text)

    for tag in soup.find_all("a", href=True):

        next_url = urljoin(url, tag["href"])

    if (
        urlparse(next_url).netloc == BASE_DOMAIN
        and next_url.startswith(BASE_PATH)
        and next_url not in visited_urls
    ):
        scrape(next_url)

start_url = "https://www.musikwissenschaft.uni-wuerzburg.de/musikinstrumente/bestand/inventarliste/"
scrape(start_url)

with open("inventory_numbers.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(scraped_text))

with open("inventory_numbers.csv", "w", encoding="utf-8", newline="") as f:
    csv.writer(f, delimiter=";").writerows([entry.lstrip(", ") for entry in line.split(";", 2)] for line in scraped_text)